In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
import pickle

In [2]:
## load dataset
df = pd.read_csv('Churn_Modelling.csv.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
## preprocess the data 
## drop irrelevant columns
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], axis=1)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
## Encode categorical variables
label_encoder_gender = LabelEncoder()
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])
df.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [5]:
## oneHotEncode multiple categorical variables
one_hot_encoder_geography = OneHotEncoder()
encoded_geo = one_hot_encoder_geography.fit_transform(df[['Geography']])
encoded_geo_df = pd.DataFrame(encoded_geo.toarray(), columns= one_hot_encoder_geography.get_feature_names_out(['Geography']))

In [6]:
df = pd.concat([df.drop('Geography', axis=1), encoded_geo_df], axis=1)
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [7]:
## pickle encoders
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('one_hot_encoder_geography.pkl', 'wb') as file:
    pickle.dump(one_hot_encoder_geography,file)



In [8]:
## separating X and y, train test split, scale the Xs.

X = df.drop('Exited', axis=1)
y= df[['Exited']]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [9]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard


In [11]:
## model
model = Sequential([
    Dense(32, activation = 'relu', input_shape = (X_train.shape[1],)),
    Dense(1, activation='sigmoid')
])

In [12]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                416       
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 449 (1.75 KB)
Trainable params: 449 (1.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [13]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()


In [14]:
## compile
model.compile(optimizer=opt, loss= loss, metrics = ['accuracy'])

In [15]:
## TensorBoard
import datetime
logs_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir = logs_dir, histogram_freq=1)

In [16]:
## EarlyStopping
earlystopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [17]:
##    train
history = model.fit(
    X_train, y_train, 
    validation_data = (X_test, y_test),
    epochs=50,
    callbacks = (tensorboard_callback)
)

Epoch 1/50


250/250 [==============================] - 4s 6ms/step - loss: 0.4075 - accuracy: 0.8304 - val_loss: 0.3556 - val_accuracy: 0.8525
Epoch 2/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3601 - accuracy: 0.8496 - val_loss: 0.3512 - val_accuracy: 0.8580
Epoch 3/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3521 - accuracy: 0.8546 - val_loss: 0.3472 - val_accuracy: 0.8605
Epoch 4/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3503 - accuracy: 0.8544 - val_loss: 0.3499 - val_accuracy: 0.8585
Epoch 5/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3463 - accuracy: 0.8579 - val_loss: 0.3521 - val_accuracy: 0.8590
Epoch 6/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3425 - accuracy: 0.8590 - val_loss: 0.3434 - val_accuracy: 0.8595
Epoch 7/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3406 - accuracy: 0.8596 - val_loss: 0.3454 - val_accuracy: 0.8565
Epoc

In [21]:
model.save('model.h5')

In [19]:
## load tensorboard
%load_ext tensorboard

In [23]:
%tensorboard --logdir logs/fit/20250627-182232

Reusing TensorBoard on port 6007 (pid 17560), started 0:00:06 ago. (Use '!kill 17560' to kill it.)